# Wake Word Leaderboard — what the plugin arena measured

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/ww_leaderboard.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/benchmarks/ww_leaderboard.ipynb)

## Start here

This notebook reads every wake-word test result that the [OVOS plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena) published on Hugging Face. It scores the results again with the arena's own code. Then it shows which plugin wakes on its phrase, and which plugin wakes by mistake. It needs no GPU and no plugins, only the internet, and it takes about two minutes. Press **Run All** and read "What you got" at the bottom. To measure the plugins yourself, use [`ww_benchmark.ipynb`](ww_benchmark.ipynb).

Change the settings in the Settings cell, and then run all cells. To change a setting without a change to the notebook, set an environment variable with the same name. You can also write the line `NAME=value` in a file called `.env`, in the folder where the notebook runs.

In [ ]:
#@title Code that reads the settings
# A setting comes from an environment variable of the same name, else from a line NAME=value in the file .env,
# else from the value in the Settings cell. A value from outside is turned into the type of that value.
import json, os

_DOTENV = {}
if os.path.isfile(".env"):
    with open(".env", encoding="utf-8") as _f:
        for _line in _f:
            _name, _sep, _value = _line.strip().partition("=")
            if _sep and not _name.startswith("#"):
                _DOTENV[_name.strip()] = _value.strip().strip("\"'")


def setting(name, default):
    raw = os.environ.get(name, _DOTENV.get(name))
    if raw is None:
        return default
    if isinstance(default, bool):
        if raw.strip().lower() in ("1", "true", "yes", "on"):
            return True
        if raw.strip().lower() in ("0", "false", "no", "off", ""):
            return False
        raise ValueError(f"{name} must be True or False, not {raw!r}")
    if isinstance(default, (int, float)):
        try:
            return type(default)(raw)
        except ValueError:
            raise ValueError(f"{name} must be a number, not {raw!r}") from None
    if isinstance(default, (list, dict)):
        return json.loads(raw)
    return raw


In [ ]:
# Settings: change the values below.
DATASETS = setting("DATASETS", [])                    # arena test sets to read, such as ["picovoice-computer"]; empty: every published wake-word set
SAME_PHRASE_ONLY = setting("SAME_PHRASE_ONLY", True)  # True: a detector is scored only on the test sets of its own phrase

# Quick test
SMOKE = setting("SMOKE", False)                       # True: reads two test sets only
# N_SAMPLES and EPOCHS are not used: this notebook reads results and runs no model.

# ── Advanced: leave these as they are unless you know you need them ──
ARENA_REF = setting("ARENA_REF", "72a05099004d652daf9f01a8fa0d08ebf201e2fc")   # the plugin arena commit whose registry and scoring code are used
WORK_DIR = setting("WORK_DIR", "")                    # the folder for downloads; empty: ./ww_leaderboard_work, on Kaggle /tmp/ww_leaderboard_work
OUTPUT_DIR = setting("OUTPUT_DIR", "./ww_leaderboard_output")   # the tables and the plots go here
SHOW_DETAILS = setting("SHOW_DETAILS", False)         # True: also print the tables behind every step

In [ ]:
import os

WORK_DIR = WORK_DIR or ("/tmp/ww_leaderboard_work" if os.path.isdir("/kaggle") else "./ww_leaderboard_work")
SMOKE_DATASETS = ["synthetic-wakewords-hey_mycroft", "picovoice-computer"]

## Step 1 — Install and get the arena code

Installs the few packages the notebook needs and downloads one fixed commit of the plugin arena: its list of plugins and test sets, and the code that scores them.

In [ ]:
#@title Code for step 1: the step lines, the internet check and the arena checkout
import json, os, re, subprocess, sys, time, urllib.error, urllib.request
from pathlib import Path

def detail(*a, **k):
    # SHOW_DETAILS: a line of the tables behind a step, with the clock time
    if SHOW_DETAILS:
        print(time.strftime("%H:%M:%S"), *a, **k)
if not SHOW_DETAILS:
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars; the step lines say what happened
_STEPS, _last, _done = {}, [time.time()], [time.time()]
def tick(name):
    now = time.time()
    _STEPS[name], _last[0] = now - _last[0], now
def done(name, message):
    # the step's one line, with the clock time and the seconds the step took
    tick(name)
    print(f"{message} [{time.strftime('%H:%M:%S')}, {_last[0] - _done[0]:.0f} s]")
    _done[0] = _last[0]

def check_internet():
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=15)
    except (urllib.error.URLError, OSError) as exc:
        raise SystemExit("No internet. On Kaggle, open the Settings panel and switch Internet on. "
                         f"({exc})") from None

ARENA_URL = "https://github.com/OpenVoiceOS/ovos-plugin-arena"

def fetch_arena(ref, folder):
    # one commit of the plugin arena: its registry, its runner and its scoring code
    folder = Path(folder)
    def git(*args, check=True):
        return subprocess.run(["git", "-C", str(folder), *args], check=check,
                              capture_output=True, text=True).stdout.strip()
    if not (folder / ".git").is_dir():
        folder.mkdir(parents=True, exist_ok=True)
        git("init", "-q")
        git("remote", "add", "origin", ARENA_URL)
    if git("rev-parse", "-q", "--verify", "HEAD", check=False) != ref:
        git("fetch", "-q", "--depth", "1", "origin", ref)
        git("checkout", "-q", "--force", "FETCH_HEAD")
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
    return git("rev-parse", "HEAD")

def phrase(text):
    # "hey_mycroft", "Hey Mycroft" and "hey-mycroft" are the same phrase
    return re.sub(r"[\s_\-]+", " ", (text or "").lower()).strip()

# The dataset names its wake-word folder; for these sets the folder name is not the phrase.
DATASET_PHRASE = {"sam-wake-word": "sam"}

def dataset_phrase(ds):
    return phrase(DATASET_PHRASE.get(ds.dataset_id, ds.wakeword))

def fighter_phrase(comp):
    return phrase(next(iter(comp.config.get("hotwords", {})), ""))

In [ ]:
check_internet()
%pip install --quiet --pre "pydantic>=2" "huggingface_hub>=0.30" "pyarrow>=15" "pandas>=2" "matplotlib>=3.8"
ARENA_DIR = Path(WORK_DIR) / "ovos-plugin-arena"
ARENA_SHA = fetch_arena(ARENA_REF, ARENA_DIR)
done("install", f"Step 1 done: plugin arena at commit {ARENA_SHA[:10]}.")

## Step 2 — Read the published results

Each arena test set has one Hugging Face dataset of results, named `OpenVoiceOS/ovos-wake-word-bench-<test set>`. A row is one audio clip given to one plugin: the clip has the phrase or not, and the plugin woke or not. Reading needs no token.

<details>
<summary>The test sets</summary>

- `picovoice-*`: people recorded by Picovoice for its open benchmark. This is the fairest test, because no plugin trained on them.
- `community-*`: people from the OVOS community. A plugin that trained on one of these sets is not scored on it.
- `synthetic-wakewords-*`: the phrase in many computer voices, with no room echo and no microphone.
- `sam-wake-word`: the phrase "sam", with its own hard negatives.

The clips that must not wake a plugin come from speech, sounds, music and noise in the TigreGotico not-wake-word sets and from the Multilingual Spoken Words corpus.

</details>

In [ ]:
#@title Code for step 2: reads the result rows of each test set
import logging
logging.getLogger().setLevel(logging.WARNING if SHOW_DETAILS else logging.ERROR)

from registry.loaders import list_competitors, list_datasets, results_repo_for
from arena.predictions import group_rows, load_predictions

FIGHTERS = {c.competitor_id: c for c in list_competitors("wake_word")}

def wanted_datasets():
    names = SMOKE_DATASETS if SMOKE else DATASETS
    sets = [d for d in list_datasets("wake_word") if d.role == "eval"]
    if names:
        unknown = sorted(set(names) - {d.dataset_id for d in sets})
        assert not unknown, f"DATASETS names {unknown}, which the arena does not know. Known: {sorted(d.dataset_id for d in sets)}"
        sets = [d for d in sets if d.dataset_id in names]
    return sets

def read_results(ds, revision="main"):
    # (rows grouped by sample and fighter, rows of fighters the registry no longer has, rows dropped for training overlap)
    repo = results_repo_for("wake_word", ds.dataset_id)
    rows = load_predictions(repo, revision=revision, lang=ds.lang)
    unregistered, trained_on = {}, {}
    grouped = group_rows(rows, unregistered=unregistered, trained_on_dropped=trained_on)
    by_fighter = {}
    for (_, dataset_id, lang), samples in grouped.items():
        if dataset_id == ds.dataset_id and lang == ds.lang:
            for sample_rows in samples.values():
                for competitor_id, row in sample_rows.items():
                    by_fighter.setdefault(competitor_id, []).append(row)
    return by_fighter, unregistered, trained_on

In [ ]:
DATASET_DEFS = wanted_datasets()
RESULTS, MISSING, STALE = {}, [], {}
for ds in DATASET_DEFS:
    by_fighter, unregistered, trained_on = read_results(ds)
    if not by_fighter:
        MISSING.append(ds.dataset_id)
        continue
    RESULTS[ds.dataset_id] = by_fighter
    if unregistered or trained_on:
        STALE[ds.dataset_id] = {**unregistered, **trained_on}
    detail(ds.dataset_id, {k: len(v) for k, v in by_fighter.items()})
n_rows = sum(len(r) for f in RESULTS.values() for r in f.values())
done("read", f"Step 2 done: {n_rows} result rows from {len(RESULTS)} of {len(DATASET_DEFS)} test sets"
     + (f"; no results yet for {', '.join(MISSING)}" if MISSING else "") + ".")

## Step 3 — Score every plugin

The arena's own scoring code turns the rows into one board per test set. The **error rate** is the share of clips the plugin got wrong. A false reject is a clip with the phrase that did not wake the plugin. A false accept is a clip without the phrase that woke it. Lower is better. A plugin with fewer than 30 clips on a test set is shown but not ranked, because so few clips cannot tell two plugins apart.

In [ ]:
#@title Code for step 3: one board per test set, as the arena builds it
import pandas as pd
from arena.metrics import MIN_BOARD_SAMPLES, build_benchmark_board, ww_reference

def board_table(ds, by_fighter):
    if SAME_PHRASE_ONLY:
        by_fighter = {cid: rows for cid, rows in by_fighter.items()
                      if cid in FIGHTERS and fighter_phrase(FIGHTERS[cid]) == dataset_phrase(ds)}
    if not by_fighter:
        return [], None
    board = build_benchmark_board("wake_word", ds.dataset_id, ds.lang, by_fighter, "notebook",
                                  dataset_revision=ds.source.revision)
    table = []
    for e in board.entries:
        rows = by_fighter[e.competitor_id]
        positives = sum(1 for r in rows if ww_reference(r) is True)
        m = e.metrics
        table.append({
            "test set": ds.dataset_id, "fighter": e.competitor_id, "plugin": e.plugin_id,
            "clips": e.samples, "with phrase": positives, "without phrase": e.samples - positives,
            "error rate": m.get("error_rate"), "error low": e.primary_metric_ci_lower,
            "error high": e.primary_metric_ci_upper,
            "false accepts": m.get("false_accept_rate"), "false rejects": m.get("false_reject_rate"),
            "false wake-ups per hour": m.get("fa_per_hour"), "latency ms": m.get("latency_ms_median"),
            "ranked": not e.unranked, "why not ranked": e.unranked_reason or "",
        })
    return table, board

In [ ]:
BOARDS, rows = {}, []
for ds in DATASET_DEFS:
    if ds.dataset_id in RESULTS:
        table, board = board_table(ds, RESULTS[ds.dataset_id])
        rows += table
        BOARDS[ds.dataset_id] = board
LEADERBOARD = pd.DataFrame(rows)
assert len(LEADERBOARD), "No plugin has a result on these test sets. Set SAME_PHRASE_ONLY = False to see every row."
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
LEADERBOARD.to_csv(Path(OUTPUT_DIR) / "ww_leaderboard.csv", index=False)
n_ranked = int(LEADERBOARD["ranked"].sum())
if SHOW_DETAILS:
    print(LEADERBOARD.to_string(index=False))
done("score", f"Step 3 done: {len(LEADERBOARD)} plugin results on {LEADERBOARD['test set'].nunique()} test sets, "
     f"{n_ranked} with {MIN_BOARD_SAMPLES} or more clips.")

## Step 4 — Check against the arena's own boards

The arena commits each board it builds to its website data. This step reads the result rows at the revision that the board names. It scores them again and compares every number. A difference means the scoring code or the rows changed since the board was built.

In [ ]:
#@title Code for step 4: scores each committed board's rows again and compares the numbers
def committed_board(ds):
    path = ARENA_DIR / "frontend-static/public/data" / f"benchmark-wake_word-{ds.dataset_id}-{ds.lang}.json"
    return json.loads(path.read_text()) if path.is_file() else None

def recheck(ds):
    # (fighters compared, fighters whose numbers differ, the revision read)
    committed = committed_board(ds)
    if not committed or not committed.get("predictions_revisions"):
        return 0, [], None
    revision = next(iter(committed["predictions_revisions"].values()))
    by_fighter, _, _ = read_results(ds, revision)
    if not by_fighter:
        return 0, [], revision
    rebuilt = build_benchmark_board("wake_word", ds.dataset_id, ds.lang, by_fighter, "notebook",
                                    dataset_revision=ds.source.revision)
    mine = {e.competitor_id: (e.samples, e.metrics) for e in rebuilt.entries}
    theirs = {e["competitor_id"]: (e["samples"], e["metrics"]) for e in committed["entries"]}
    differ = [cid for cid in theirs if mine.get(cid) != theirs[cid]]
    return len(theirs), differ, revision

In [ ]:
CHECKS = {ds.dataset_id: recheck(ds) for ds in DATASET_DEFS if ds.dataset_id in RESULTS}
compared = sum(n for n, _, _ in CHECKS.values())
differ = {k: d for k, (_, d, _) in CHECKS.items() if d}
for dataset_id, (n, d, revision) in CHECKS.items():
    detail(dataset_id, f"revision {revision}", f"{n} compared", f"differ: {d}" if d else "all equal")
done("check", f"Step 4 done: {compared} committed board entries compared, "
     + (f"{sum(map(len, differ.values()))} differ: {differ}." if differ else "every number is equal."))

## Step 5 — Plots

The first plot shows each plugin result as a point. A point to the left wakes less by mistake. A point lower down misses the phrase less. A filled point has 30 or more clips. The second plot shows how long each plugin takes to decide on one clip.

In [ ]:
#@title Code for step 5: the plots
import matplotlib
import matplotlib.pyplot as plt

def plot_errors(table, path):
    fig, ax = plt.subplots(figsize=(8, 6))
    colours = dict(zip(sorted(table["plugin"].unique()), matplotlib.colormaps["tab10"].colors))
    for plugin, part in table.groupby("plugin"):
        for ranked, marker_face in ((True, colours[plugin]), (False, "none")):
            sub = part[part["ranked"] == ranked]
            ax.scatter(sub["false accepts"], sub["false rejects"], s=60, edgecolors=colours[plugin],
                       facecolors=marker_face, label=plugin if ranked or part["ranked"].sum() == 0 else None)
    ax.set_xlabel("false accepts (share of clips without the phrase that woke it)")
    ax.set_ylabel("false rejects (share of clips with the phrase it missed)")
    ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
    ax.legend(fontsize=8); ax.set_title("Wake-word plugins: mistakes per test set")
    fig.tight_layout(); fig.savefig(path, dpi=120)
    return fig

def plot_latency(table, path):
    fig, ax = plt.subplots(figsize=(8, 4))
    per = table.groupby("plugin")["latency ms"].median().sort_values()
    ax.barh(per.index, per.values)
    ax.set_xlabel("median time to decide on one clip (ms)")
    fig.tight_layout(); fig.savefig(path, dpi=120)
    return fig

In [ ]:
plot_errors(LEADERBOARD, Path(OUTPUT_DIR) / "ww_errors.png"); plt.show()
plot_latency(LEADERBOARD, Path(OUTPUT_DIR) / "ww_latency.png"); plt.show()
done("plots", f"Step 5 done: plots saved in {OUTPUT_DIR}.")

## What you got

**The table** below lists the best plugin result on each test set, and the whole table is in `ww_leaderboard_output/ww_leaderboard.csv`. **The plots** are `ww_errors.png` and `ww_latency.png` in the same folder.

Read the error rate together with the number of clips, and check the column `without phrase`: a result with no clips without the phrase says nothing about false wake-ups. A board with few clips ranks nobody, and the arena fills it as it runs more tests. To add clips, run [`ww_benchmark.ipynb`](ww_benchmark.ipynb). Synthetic voices are easier than people, so a low error on a `synthetic-wakewords` set does not say much about a real home. The `picovoice` sets are the fairest test.

<details>
<summary>Read more</summary>

- [The plugin arena](https://github.com/OpenVoiceOS/ovos-plugin-arena) and its [methodology](https://github.com/OpenVoiceOS/ovos-plugin-arena/blob/dev/docs/methodology.md)
- [Wake word plugins](https://openvoiceos.github.io/ovos-technical-manual/312-wake_word_plugins/) in the OVOS manual
- [`docs/benchmarks.md`](../docs/benchmarks.md), the guide to these notebooks

</details>

In [ ]:
best = (LEADERBOARD[LEADERBOARD["ranked"]].sort_values("error rate").groupby("test set").head(1)
        if n_ranked else LEADERBOARD.sort_values("error rate").groupby("test set").head(1))
columns = ["test set", "fighter", "clips", "error rate", "false accepts", "false rejects"]
print(f"""Test sets read        : {len(RESULTS)} of {len(DATASET_DEFS)}{' (no results yet: ' + ', '.join(MISSING) + ')' if MISSING else ''}
Plugin results        : {len(LEADERBOARD)}, {n_ranked} with enough clips to rank
Check against arena   : {compared} entries compared, {sum(map(len, differ.values()))} differ
No negative clips    : {int((LEADERBOARD["without phrase"] == 0).sum())} of {len(LEADERBOARD)} results (their false accept rate is unknown)
Rows of old fighters  : {sum(sum(v.values()) for v in STALE.values())} (fighters the arena no longer lists; not scored)
Arena commit          : {ARENA_SHA}
Minutes per step      : {', '.join(f'{k} {v / 60:.1f}' for k, v in _STEPS.items())}

{'Best ranked result' if n_ranked else 'Best result (none has enough clips to rank)'} per test set:
{best[columns].to_string(index=False)}""")
if SMOKE:   # the trial must read real rows and reproduce the arena's numbers
    assert len(RESULTS) == len(SMOKE_DATASETS), f"missing results: {MISSING}"
    assert compared and not differ, f"the scores differ from the arena's boards: {differ}"

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.